# ML4G Project 1: Getting Started

This notebook shows you how to **load the competition data** on Kaggle and how to **write a correctly formatted submission file**. The submission it produces is a constant placeholder, which scores 0. Your solution goes in the marked section.

**Before you start:** you may need to add the competition dataset to this notebook (right panel → *Add Input*), then run all cells.

## 1. Setup

In [1]:
import os
import glob
import zipfile

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import PolynomialFeatures

## 2. Locate the data

Kaggle mounts attached datasets read-only under `/kaggle/input/`. If any files are still inside `.zip` archives, the cell below extracts them to `/kaggle/working/data`.

In [2]:
INPUT_ROOT = "/kaggle/input"
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
EXTRACT_DIR = os.path.join(WORK_DIR, "data")

for zp in glob.glob(os.path.join(INPUT_ROOT, "**", "*.zip"), recursive=True):
    target = os.path.join(EXTRACT_DIR, os.path.splitext(os.path.basename(zp))[0])
    if not os.path.isdir(target):
        with zipfile.ZipFile(zp) as zf:
            zf.extractall(target)

ALL_FILES = sorted(
    p for root in [INPUT_ROOT, EXTRACT_DIR]
    for p in glob.glob(os.path.join(root, "**", "*"), recursive=True)
    if os.path.isfile(p) and not p.endswith(".zip")
)
print(f"{len(ALL_FILES)} files found")
for p in ALL_FILES:
    print(p)

57 files found
/kaggle/input/competitions/ml-4-g-project-1/CAGE-train/CAGE-train/X1_train_info.tsv
/kaggle/input/competitions/ml-4-g-project-1/CAGE-train/CAGE-train/X1_train_y.tsv
/kaggle/input/competitions/ml-4-g-project-1/CAGE-train/CAGE-train/X1_val_info.tsv
/kaggle/input/competitions/ml-4-g-project-1/CAGE-train/CAGE-train/X1_val_y.tsv
/kaggle/input/competitions/ml-4-g-project-1/CAGE-train/CAGE-train/X2_train_info.tsv
/kaggle/input/competitions/ml-4-g-project-1/CAGE-train/CAGE-train/X2_train_y.tsv
/kaggle/input/competitions/ml-4-g-project-1/CAGE-train/CAGE-train/X2_val_info.tsv
/kaggle/input/competitions/ml-4-g-project-1/CAGE-train/CAGE-train/X2_val_y.tsv
/kaggle/input/competitions/ml-4-g-project-1/CAGE-train/CAGE-train/X3_test_info.tsv
/kaggle/input/competitions/ml-4-g-project-1/DNase-bed/X1.bed
/kaggle/input/competitions/ml-4-g-project-1/DNase-bed/X2.bed
/kaggle/input/competitions/ml-4-g-project-1/DNase-bed/X3.bed
/kaggle/input/competitions/ml-4-g-project-1/DNase-bigwig/X1.bw
/kag

In [3]:
def find_file(name):
    # Path of the unique file whose name is `name`.
    hits = [p for p in ALL_FILES if os.path.basename(p) == name]
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected exactly 1 file named {name}, found {len(hits)}: {hits}")
    return hits[0]


def find_track(mark, cell_line, kind):
    # Path of a chromatin track. kind: 'bed' (narrowPeak peaks) or 'bigwig' (continuous signal).
    exts = (".bed", ".bed.gz", ".narrowpeak", ".narrowpeak.gz") if kind == "bed" else (".bw", ".bigwig")
    hits = [
        p for p in ALL_FILES
        if mark in p and cell_line in os.path.basename(p) and p.lower().endswith(exts)
    ]
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected exactly 1 {kind} file for {mark}/{cell_line}, found {len(hits)}: {hits}")
    return hits[0]

## 3. Gene information and targets

Each cell line and split has an `*_info.tsv` file with gene coordinates and, for X1 and X2, a `*_y.tsv` file with CAGE expression. The X3 test genes come without targets.

| Split | Chromosomes | Cell lines |
|---|---|---|
| train (set A) | chr2–13, chr15–18, chr20–22 | X1, X2 |
| val (set B) | chr14, chr19 | X1, X2 |
| test (set C) | chr1 | X3 |

In [4]:
def load_split(cell_line, split):
    info = pd.read_csv(find_file(f"{cell_line}_{split}_info.tsv"), sep="\t")
    if split == "test":
        return info
    y = pd.read_csv(find_file(f"{cell_line}_{split}_y.tsv"), sep="\t")
    return info.merge(y, on="gene_name", how="inner", validate="one_to_one")

data = {
    ("X1", "train"): load_split("X1", "train"),
    ("X1", "val"):   load_split("X1", "val"),
    ("X2", "train"): load_split("X2", "train"),
    ("X2", "val"):   load_split("X2", "val"),
    ("X3", "test"):  load_split("X3", "test"),
}

for key, df in data.items():
    print(key, df.shape)
data[("X1", "train")].head()

('X1', 'train') (14310, 8)
('X1', 'val') (1974, 8)
('X2', 'train') (14310, 8)
('X2', 'val') (1974, 8)
('X3', 'test') (1984, 7)


,gene_name,chr,gene_start,gene_end,TSS_start,TSS_end,strand,gex
0,SLC20A1,chr2,112645939,112663825,112658362,112658412,+,0.000000
1,C11orf58,chr11,16613132,16758340,16738643,16738693,+,2239.103328
2,ZSCAN9,chr6,28224886,28233487,28225263,28225313,+,19.798064
3,CD19,chr16,28931965,28939342,28931956,28932006,+,411.530623
4,TMEM123,chr11,102396332,102470384,102452789,102452839,-,34.214129


## 4. Chromatin tracks

Each assay (DNase, H3K4me3, H3K4me1, H3K27ac, H3K36me3, H3K27me3, H3K9me3) comes as **BED narrowPeak** (called peaks) and **bigWig** (continuous signal), one file per cell line.

narrowPeak files have 10 tab-separated columns without a header. Format specification: https://genome.ucsc.edu/FAQ/FAQformat.html#format12

In [5]:
NARROWPEAK_COLS = ["chrom", "chromStart", "chromEnd", "name", "score", "strand",
                   "signalValue", "pValue", "qValue", "peak"]

def load_peaks(mark, cell_line):
    return pd.read_csv(find_track(mark, cell_line, "bed"), sep="\t", header=None,
                       names=NARROWPEAK_COLS, usecols=range(10), comment="#")

peaks = load_peaks("DNase", "X1")
print(peaks.shape)
peaks.head()

(56452, 10)


,chrom,chromStart,chromEnd,name,score,strand,signalValue,pValue,qValue,peak
0,chr1,181405,181555,.,0,.,12.379543,-1,-1,75
1,chr1,267990,268140,.,0,.,12.379543,-1,-1,75
2,chr1,629160,629310,.,0,.,22.263896,-1,-1,75
3,chr1,629494,629644,.,0,.,10.732151,-1,-1,75
4,chr1,629870,630020,.,0,.,3603.694491,-1,-1,75


Reading bigWig files requires `pyBigWig`, which needs *Internet on* in the notebook settings (phone-verified Kaggle account):

In [6]:
!pip install -q pyBigWig
import pyBigWig

bw = pyBigWig.open(find_track("DNase", "X1", "bigwig"))
print(bw.chroms("chr2"))
print(bw.stats("chr2", 1_000_000, 1_002_000, type="mean"))   # mean signal in a 2 kb window

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.1/189.1 kB 3.5 MB/s eta 0:00:00
242193529
[0.04259009861318022]


## 5. Your solution

Build your features and model here. Train and validate on X1 and X2, then predict the expression of the X3 test genes.

The submission is scored with **Spearman's rank correlation**, so only the order of your predictions matters. You can compute it locally with `scipy.stats.spearmanr` on your validation sets.

At the end of this section, `pred` must be a numeric array with one value per row of `test_genes`, in the same order.

In [7]:
# function definitions for building features

MARKS = [
    "DNase", "H3K4me3", "H3K4me1",
    "H3K27ac", "H3K36me3",
    "H3K27me3", "H3K9me3"
]

def mean_signal(bw, chrom, start, end):
    chrom_sizes = bw.chroms()

    start = max(0, int(start))
    end = min(int(end), chrom_sizes[chrom])

    if end <= start:
        return np.nan

    value = bw.stats(
        chrom, start, end,
        type="mean",
        exact=True
    )[0]

    return np.nan if value is None else value


def extract_features(df, cell_line):

    features = pd.DataFrame(index=df.index)

    for mark in MARKS:

        bw = pyBigWig.open(
            find_track(mark, cell_line, "bigwig")
        )

        try:
            for window in [50, 200, 1000, 5000, 20000]:

                values = []

                for _, gene in tqdm(
                    df.iterrows(),
                    total=len(df),
                    desc=f"{cell_line} {mark}",
                    leave=False
                ):

                    chrom = gene["chr"]
                    tss = (
                        int(gene["TSS_start"]) +
                        int(gene["TSS_end"])
                    ) // 2

                    signal = mean_signal(
                        bw,
                        chrom,
                        tss - window,
                        tss + window
                    )

                    values.append(signal)

                features[f"{mark}_tss_pm{window}"] = values

            # Gene body: TSS to the annotated 3' end
            values = []

            for _, gene in tqdm(
                    df.iterrows(),
                    total=len(df),
                    desc=f"{cell_line} {mark}",
                    leave=False
                ):

                chrom = gene["chr"]
                tss = (
                    int(gene["TSS_start"]) +
                    int(gene["TSS_end"])
                ) // 2

                if gene["strand"] == "+":
                    start, end = tss, int(gene["gene_end"])
                else:
                    start, end = int(gene["gene_start"]), tss

                signal = mean_signal(bw, chrom, start, end)

                values.append(signal)

            features[f"{mark}_body"] = values

        finally:
            bw.close()

    return features

In [8]:
# building features

X1_train = extract_features(data[("X1", "train")], "X1")
X2_train = extract_features(data[("X2", "train")], "X2")
X1_val = extract_features(data[("X1", "val")], "X1")
X2_val = extract_features(data[("X2", "val")], "X2")

X1 DNase:   0%|          | 0/14310 [00:00<?, ?it/s]

X1 DNase:   0%|          | 0/14310 [00:00<?, ?it/s]

X1 DNase:   0%|          | 0/14310 [00:00<?, ?it/s]

X1 DNase:   0%|          | 0/14310 [00:00<?, ?it/s]

X1 DNase:   0%|          | 0/14310 [00:00<?, ?it/s]

...

X2 H3K9me3:   0%|          | 0/1974 [00:00<?, ?it/s]

X2 H3K9me3:   0%|          | 0/1974 [00:00<?, ?it/s]

X2 H3K9me3:   0%|          | 0/1974 [00:00<?, ?it/s]

X2 H3K9me3:   0%|          | 0/1974 [00:00<?, ?it/s]

X2 H3K9me3:   0%|          | 0/1974 [00:00<?, ?it/s]

In [9]:
# concatenate data into training and validation sets

X_train = pd.concat(
    [X1_train, X2_train],
    ignore_index=True
)

y_train = pd.concat(
    [
        data[("X1", "train")]["gex"],
        data[("X2", "train")]["gex"]
    ],
    ignore_index=True
)

X_val = pd.concat(
    [X1_val, X2_val],
    ignore_index=True
)

y_val = pd.concat(
    [
        data[("X1", "val")]["gex"],
        data[("X2", "val")]["gex"]
    ],
    ignore_index=True
)

In [10]:
# log transform 

X_train_log = np.log1p(X_train.clip(lower=0))
X_val_log = np.log1p(X_val.clip(lower=0))

y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

In [11]:
# model (ridge regression)

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    Ridge(alpha=100.0)
)

In [12]:
# cross-validation

chromosomes = pd.concat([
    data[("X1", "train")]["chr"],
    data[("X2", "train")]["chr"]
], ignore_index=True)

cell_lines = np.array(
    ["X1"] * len(X1_train) +
    ["X2"] * len(X2_train)
)

cv = GroupKFold(n_splits=5)

results = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train_log, y_train_log, chromosomes)
):

    for train_cl, val_cl in [("X1", "X2"), ("X2", "X1")]:

        tr = train_idx[cell_lines[train_idx] == train_cl]
        va = val_idx[cell_lines[val_idx] == val_cl]

        m = clone(model)

        m.fit(
            X_train_log.iloc[tr],
            y_train_log.iloc[tr]
        )

        pred = m.predict(X_train_log.iloc[va])

        rho = spearmanr(
            y_train.iloc[va],
            pred
        ).statistic

        results.append({
            "fold": fold + 1,
            "direction": f"{train_cl} → {val_cl}",
            "spearman": rho
        })

results_df = pd.DataFrame(results)

print(results_df)

print(
    results_df.groupby("direction")["spearman"].mean()
)

print(
    "Overall mean:",
    results_df["spearman"].mean()
)

   fold direction  spearman
0     1   X1 → X2  0.777338
1     1   X2 → X1  0.728772
2     2   X1 → X2  0.754253
3     2   X2 → X1  0.731310
4     3   X1 → X2  0.773511
5     3   X2 → X1  0.729994
6     4   X1 → X2  0.786814
7     4   X2 → X1  0.752776
8     5   X1 → X2  0.772729
9     5   X2 → X1  0.747892
direction
X1 → X2    0.772929
X2 → X1    0.738149
Name: spearman, dtype: float64
Overall mean: 0.7555388892499011


In [13]:
# final model

model.fit(X_train_log, y_train_log)

pred = model.predict(X_val_log)

rho = spearmanr(y_val_log, pred).statistic

print("Validation Spearman:", rho)

Validation Spearman: 0.7791826682930496


In [14]:
# inspect the most important weights

ridge = model.named_steps["ridge"]

coef_df = pd.DataFrame({
    "feature": X_train_log.columns,
    "coefficient": ridge.coef_
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

coef_df = coef_df.sort_values(
    "abs_coefficient",
    ascending=False
)

print(coef_df)

                 feature  coefficient  abs_coefficient
2       DNase_tss_pm1000    -0.849184         0.849184
0         DNase_tss_pm50     0.801615         0.801615
19     H3K27ac_tss_pm200     0.697575         0.697575
1        DNase_tss_pm200     0.466161         0.466161
18      H3K27ac_tss_pm50    -0.417124         0.417124
29         H3K36me3_body     0.313362         0.313362
8     H3K4me3_tss_pm1000     0.306655         0.306655
20    H3K27ac_tss_pm1000     0.271320         0.271320
3       DNase_tss_pm5000    -0.195612         0.195612
9     H3K4me3_tss_pm5000     0.163576         0.163576
6       H3K4me3_tss_pm50     0.144856         0.144856
7      H3K4me3_tss_pm200    -0.137479         0.137479
26   H3K36me3_tss_pm1000    -0.116365         0.116365
13     H3K4me1_tss_pm200    -0.114932         0.114932
4      DNase_tss_pm20000     0.114673         0.114673
11          H3K4me3_body     0.114145         0.114145
32   H3K27me3_tss_pm1000    -0.113674         0.113674
31    H3K2

In [15]:
# final prediction

test_genes = extract_features(data[("X3", "test")], "X3")
test_genes = np.log1p(test_genes.clip(lower=0))
test_genes["gene_name"] = data[("X3", "test")]["gene_name"]

pred = model.predict(test_genes[X_train_log.columns])

X3 DNase:   0%|          | 0/1984 [00:00<?, ?it/s]

X3 DNase:   0%|          | 0/1984 [00:00<?, ?it/s]

X3 DNase:   0%|          | 0/1984 [00:00<?, ?it/s]

X3 DNase:   0%|          | 0/1984 [00:00<?, ?it/s]

X3 DNase:   0%|          | 0/1984 [00:00<?, ?it/s]

...

X3 H3K9me3:   0%|          | 0/1984 [00:00<?, ?it/s]

X3 H3K9me3:   0%|          | 0/1984 [00:00<?, ?it/s]

X3 H3K9me3:   0%|          | 0/1984 [00:00<?, ?it/s]

X3 H3K9me3:   0%|          | 0/1984 [00:00<?, ?it/s]

X3 H3K9me3:   0%|          | 0/1984 [00:00<?, ?it/s]

## 6. Write the submission

The required format is a CSV with header `gene_name,gex_predicted` and exactly one row per gene in `X3_test_info.tsv`, **without an index column**. The checks below catch the most common mistakes before you spend one of your 10 submissions.

In [16]:
submission = pd.DataFrame({
    "gene_name": test_genes["gene_name"].to_numpy(),
    "gex_predicted": np.asarray(pred, dtype=float),
})

assert len(submission) == len(test_genes), "One prediction per test gene required"
assert submission["gene_name"].is_unique, "Duplicate gene names"
assert np.isfinite(submission["gex_predicted"]).all(), "Predictions contain NaN or inf"

out_path = os.path.join(WORK_DIR, "submission.csv")
submission.to_csv(out_path, index=False)   # index=False is essential
print(f"Saved {out_path}")
print(open(out_path).read()[:150])

Saved /kaggle/working/submission.csv
gene_name,gex_predicted
CAPN9,0.03734852115128273
ILF2,0.4751727003837467
ST6GALNAC5,-0.09887444321794159
MROH7-TTC4,0.27681001462365007
AGO4,0.008904


### How to submit

- **From this notebook:** click *Save Version* (Save & Run All). When it finishes, open the version's *Output* tab and click *Submit*. The competition must be attached as an input for the button to appear.
- **Manually:** download `submission.csv` from the *Output* panel and upload it on the competition's *Submit Prediction* page.

Every successful submission counts toward your team's limit of 10. Files rejected with a format error don't count.